# 4.7 IC Grouping Data

In this notebook, I perform grouping and aggregation tasks on the `ords_prods_merge` dataframe, including:
- Calculating aggregated means,
- Creating a loyalty flag,
- Comparing spending habits,
- Creating spending and order frequency flags.

In [1]:
# 1. Import libraries and data
import pandas as pd
import os

# Set base path
path = r"/Users/a/Documents/Instacart Basket Analysis"

# Path to merged dataframe WITH derived variables from 4.6
merge_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_derived.pkl")

# Import dataframe
df_ords_prods = pd.read_pickle(merge_path)

# Quick check
df_ords_prods.head(), df_ords_prods.shape

(   order_id  user_id  order_number  order_dow  order_hour_of_day  \
 0   2539329        1             1          2                  8   
 1   2539329        1             1          2                  8   
 2   2539329        1             1          2                  8   
 3   2539329        1             1          2                  8   
 4   2539329        1             1          2                  8   
 
    days_since_prior_order  product_id  add_to_cart_order  reordered  \
 0                     0.0         196                  1          0   
 1                     0.0       14084                  2          0   
 2                     0.0       12427                  3          0   
 3                     0.0       26088                  4          0   
 4                     0.0       26405                  5          0   
 
                               product_name  aisle_id  department_id  prices  \
 0                                     Soda      77.0            7.0  

In [6]:
# 2. Mean of order_number by department_id for the ENTIRE dataframe

dept_order_means = df_ords_prods.groupby('department_id')['order_number'].mean().reset_index()

# Quick look at the result
dept_order_means.head(), dept_order_means.shape

(   department_id  order_number
 0            1.0     15.457687
 1            2.0     17.277920
 2            3.0     17.179756
 3            4.0     17.811403
 4            5.0     15.213779,
 (21, 2))

### **Comparison: Mean Order Numbers (Subset vs. Entire Dataset)**

In the Exercise, the mean *order_number* was calculated for only a **subset** of the data (early orders).  
For this task, I calculated the aggregated mean across the **entire dataframe**, grouped by `department_id`.

**Differences observed:**

- The means in the **full dataset** are **higher** than the subset values.  
- This is because the subset contained only a small portion of earlier orders, which naturally had lower order numbers.  
- When using the full dataset, later orders (with much higher `order_number` values) increase the overall mean.  
- This confirms that calculating means on subsets can lead to biased or misleading interpretations if not compared to the entire dataset.

In [8]:
# 4. Create loyalty flag (Loyal, Regular, New) based on order_number mean per user

# Calculate mean order_number per user using transform()
df_ords_prods['mean_order_number'] = df_ords_prods.groupby('user_id')['order_number'].transform('mean')

# Create loyalty flag
df_ords_prods.loc[df_ords_prods['mean_order_number'] > 20, 'loyalty_flag'] = 'Loyal Customer'
df_ords_prods.loc[(df_ords_prods['mean_order_number'] > 10) & 
                  (df_ords_prods['mean_order_number'] <= 20), 'loyalty_flag'] = 'Regular Customer'
df_ords_prods.loc[df_ords_prods['mean_order_number'] <= 10, 'loyalty_flag'] = 'New Customer'

# Quick check
df_ords_prods['loyalty_flag'].value_counts()

loyalty_flag
New Customer        12091184
Loyal Customer      10945281
Regular Customer     9399776
Name: count, dtype: int64

In [10]:
# 5. Compare spending habits: descriptive statistics of prices by loyalty_flag

loyalty_price_stats = df_ords_prods.groupby('loyalty_flag')['prices'].describe()

loyalty_price_stats

,count,mean,std,min,25%,50%,75%,max
loyalty_flag,,,,,,,,
Loyal Customer,10944620.0,10.466139,340.945772,1.0,4.2,7.4,11.2,99999.0
New Customer,12090394.0,13.312566,592.339583,1.0,4.2,7.4,11.3,99999.0
Regular Customer,9399198.0,12.032609,510.198900,1.0,4.2,7.4,11.3,99999.0


In [12]:
# 6. Create spending flag based on average price per user

# Calculate average price per user
df_ords_prods['mean_price_per_user'] = df_ords_prods.groupby('user_id')['prices'].transform('mean')

# Create spending flag
df_ords_prods.loc[df_ords_prods['mean_price_per_user'] < 10, 'spending_flag'] = 'Low spender'
df_ords_prods.loc[df_ords_prods['mean_price_per_user'] >= 10, 'spending_flag'] = 'High spender'

# Quick check
df_ords_prods['spending_flag'].value_counts()

spending_flag
Low spender     31800694
High spender      635547
Name: count, dtype: int64

In [14]:
# 7. Create order frequency flag based on median days_since_prior_order

# Calculate median days since prior order per user
df_ords_prods['median_days_between_orders'] = df_ords_prods.groupby('user_id')['days_since_prior_order'].transform('median')

# Create frequency flag
df_ords_prods.loc[df_ords_prods['median_days_between_orders'] > 20, 'order_frequency_flag'] = 'Non-frequent customer'
df_ords_prods.loc[(df_ords_prods['median_days_between_orders'] > 10) & 
                  (df_ords_prods['median_days_between_orders'] <= 20), 'order_frequency_flag'] = 'Regular customer'
df_ords_prods.loc[df_ords_prods['median_days_between_orders'] <= 10, 'order_frequency_flag'] = 'Frequent customer'

# Quick check
df_ords_prods['order_frequency_flag'].value_counts()

order_frequency_flag
Frequent customer        22816320
Regular customer          6929809
Non-frequent customer     2690112
Name: count, dtype: int64

In [16]:
# 9. Export updated dataframe with new flags
export_path = os.path.join(path, "02 Data", "Prepared Data", "ords_prods_grouped.pkl")
df_ords_prods.to_pickle(export_path)

export_path

'/Users/a/Documents/Instacart Basket Analysis/02 Data/Prepared Data/ords_prods_grouped.pkl'